# PLANTILLA MODELIZACI?N PARA CLASIFICACI?N

**IMPORTANTE**: Recuerda hacer una copia de esta plantilla para no machacar la original.

**IMPORTANTE**: Esta plantilla est? dise?ada para una visi?n de m?ximos utilizando el framework de The Ultimate Algo Machine. Si tienes problemas de memoria o rendimiento recuerda reducir el problema mediante:

* Muestreo
* Balanceo undersampling
* Reducir el n?mero de algoritmos a testar
* Reducir el n?mero de par?metros a testar
* Usar random search y especificar un n_iter adecuado

El objetivo de esta plantilla es identificar la configuraci?n del modelo definitivo (algoritmo e hiperpar?metros) pero no llegar a este. El modelo definitivo lo entrenaremos con datos reales m?s adelante, aqui solo lo estamos probando con una muestra.

## IMPORTAR PAQUETES

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline

from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay

from sklearn.metrics import classification_report

#from sklearn.metrics import plot_precision_recall_curve
from sklearn.metrics import PrecisionRecallDisplay
import scikitplot as skplt
#from yellowbrick.classifier import discrimination_threshold

#Autocompletar r?pido
%config IPCompleter.greedy=True

#Desactivar la notaci?n cient?fica
pd.options.display.float_format = '{:.2f}'.format

#Desactivar los warnings
import warnings
warnings.filterwarnings("ignore")

## IMPORTAR LOS DATOS

Sustituir la ruta del proyecto.

In [ ]:
ruta_proyecto = 'C:/Users/110927/Desktop/EstructuraDirectorio/03_MACHINE_LEARNING/CASOS/00_LeadScoring'

Nombres de los ficheros de datos.

In [ ]:
nombre_x = 'x_preseleccionado.pickle'
nombre_y = 'y_preseleccionado.pickle'

Cargar los datos.

In [ ]:
x = pd.read_pickle(ruta_proyecto + '/02_Datos/03_Trabajo/' + nombre_x)
y = pd.read_pickle(ruta_proyecto + '/02_Datos/03_Trabajo/' + nombre_y)

## MODELIZAR

### Reservar el dataset de validacion

In [ ]:
train_x,val_x,train_y,val_y = train_test_split(x,y,test_size=0.3)

### Crear el pipe y el diccionario de algorimos, par?metros y valores a testar

Modificar para dejar solo los algoritmos que se quieran testar.

Modificar los par?metros.

In [ ]:
pipe = Pipeline([('algoritmo',RandomForestClassifier())])

grid = [{'algoritmo': [LogisticRegression()],
         'algoritmo__n_jobs': [-1],
         'algoritmo__solver': ['saga'],
         'algoritmo__penalty': ['elasticnet', 'l1', 'l2', 'none'],
         'algoritmo__C': [0,0.25,0.5,0.75,1]}]

### Optimizar los hiper par?metros

Elegir si se quiere usar grid search o random search.

Comentar la opci?n que no se vaya a usar.

####  Con grid search

In [ ]:
grid_search = GridSearchCV(estimator= pipe, 
                            param_grid = grid, 
                            cv = 3, 
                            scoring = 'roc_auc',
                            verbose = 0,
                            n_jobs = -1)
modelo = grid_search.fit(train_x,train_y)

pd.DataFrame(grid_search.cv_results_).sort_values(by = 'rank_test_score').head(7)

In [ ]:
modelo.best_estimator_.get_params

## EVALUAR

### Predecir sobre validaci?n

In [ ]:
pred = modelo.best_estimator_.predict_proba(val_x)[:, 1]

### Evaluar sobre validaci?n

In [ ]:
roc_auc_score(val_y, pred)

### Examinar el mejor modelo

In [ ]:
modelo.best_estimator_

## REPORTING DEL MODELO

### Gain Chart

In [ ]:
fig, ax = plt.subplots()

skplt.metrics.plot_cumulative_gain(val_y, modelo.best_estimator_.predict_proba(val_x), ax=ax) 

#Eliminamos la l?nea de los ceros y personalizamos la leyenda
                 
plt.legend(labels = ['Modelo','Aleatoria']);

### Lift Chart

In [ ]:
fig, ax = plt.subplots()

skplt.metrics.plot_lift_curve(val_y, modelo.best_estimator_.predict_proba(val_x), ax=ax) 

#Eliminamos la l?nea de los ceros y personalizamos la leyenda
             
plt.legend(labels = ['Modelo','Aleatoria']);

### ROC Chart

In [ ]:
fig, ax = plt.subplots()

skplt.metrics.plot_roc(val_y, modelo.best_estimator_.predict_proba(val_x), ax=ax) 

#Eliminamos la l?nea de los ceros y personalizamos la leyenda

plt.legend(labels = ['Modelo','Aleatoria']);